# Sea-ice concentration forecasting (02)Trains and evaluates three sea-ice forecasting models on the grids produced by the preprocessing pipeline (`backend/data/processed/**`):- **Model A - persistence**: tomorrow = today (evaluated first).- **Model B - random forest**: tabular per-cell features.- **Model C - ConvLSTM**: PyTorch, input `[batch, time, height, width, features]` -> `[batch, height, width, 1]`.All artefacts land in `backend/models/sea_ice/`. When only synthetic demo data is available the pipeline shouts this clearly and runs a **demo smoke test** whose numbers are code-path checks only - never real forecasting skill.

## 0. Setup

In [ ]:
import os, sys, logging
from pathlib import Path
import numpy as np, pandas as pd, json
import matplotlib
matplotlib.use("agg")
import matplotlib.pyplot as plt

BACKEND = Path.cwd().resolve()
if not (BACKEND / "ml" / "sea_ice").exists():
    BACKEND = BACKEND.parent / "backend"
os.chdir(BACKEND); sys.path.insert(0, str(BACKEND))
print("backend =", BACKEND)

from ml.sea_ice.utils import set_seed, configured_logger
configured_logger(verbose=False)
set_seed(7)
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 1. Load the preprocessed grids and build windows

In [ ]:
from ml.sea_ice.dataset import SeaIceDataset, ALL_CHANNELS

files = {
    "sea_ice":  BACKEND / "data/processed/sea_ice/sea_ice_clean.nc",
    "ocean":    BACKEND / "data/processed/ocean/ocean_clean.nc",
    "weather":  BACKEND / "data/processed/weather/weather_clean.nc",
}
for p in files.values():
    assert p.exists(), p

ds = SeaIceDataset.from_files(files, input_steps=7, horizon=1, spatial_step=2)
print("channels:", ALL_CHANNELS)
print("X :", ds.X.shape, " y :", ds.y.shape, " mask :", ds.mask.shape)
print(f"valid cells: {100*ds.mask.mean():.1f}%")

## 2. Chronological split and normalization (train-only scalers)

In [ ]:
splits = ds.split_chronological(0.7, 0.15)
print({k: v.size for k, v in splits.items()})
print("split boundaries:", [str(ds.times[i]) for i in (0, len(splits['train'])-1, ds.X.shape[0]-1)])

ds.fit_scalers(splits["train"])   # statistics seen only during train -> no leakage
ds.normalize()
pd.DataFrame({"channel": ALL_CHANNELS, "mean": ds.scalers["mean"].round(4),
             "std": ds.scalers["std"].round(4)})

## 3. Model A - persistence baseline

In [ ]:
from ml.sea_ice.baselines import evaluate_persistence
pers = evaluate_persistence(ds, splits["test"])
print(f"persistence  MAE={pers.mae:.4f}  RMSE={pers.rmse:.4f}  spatial-r={pers.spatial_correlation:.4f}")

## 4. Model B - random forest baselineFeatures are the standardized (cell x time) vectors flattened over the 7-day input window; targets are next-day concentration. Training uses only train-split cells.

In [ ]:
from ml.sea_ice.baselines import fit_random_forest, evaluate_random_forest
rf = fit_random_forest(ds, splits["train"], n_estimators=100, subsample=200000, random_state=7)
rf_res = evaluate_random_forest(rf, ds, splits["test"])
print(f"random forest  MAE={rf_res.mae:.4f}  RMSE={rf_res.rmse:.4f}  spatial-r={rf_res.spatial_correlation:.4f}")

## 5. Model C - ConvLSTMOne ConvLSTM layer (3x3, 16 hidden channels) plus a 1x1 head. Training uses masked MSE (only observable cells), an early-stopping loop, and saves the best checkpoint. The whole training run (baselines + ConvLSTM + metrics) is also available as a CLI: `python -m ml.sea_ice.train --demo --model convlstm --with-rf`. For a real run the `--demo` flag must be dropped and real files supplied.

In [ ]:
from ml.sea_ice.train import TrainConfig, run as train_run

cfg = TrainConfig(
    out_dir="models/sea_ice/notebook", demo=True, model="convlstm", with_rf=True,
    spatial_step=2, epochs=20, batch_size=4, seed=7,
)
summary = train_run(cfg)

In [ ]:
import pandas as pd
from ml.sea_ice.utils import plot_loss_curve
plot_loss_curve(BACKEND / cfg.out_dir / "loss_history.csv",
                BACKEND / "models/sea_ice/notebook/notebook_loss.png")
pd.read_csv(BACKEND / cfg.out_dir / "loss_history.csv").tail(5)

## 6. Evaluation (test split)

In [ ]:
from ml.sea_ice.evaluate import run as eval_run
ev = eval_run(Path(cfg.out_dir))
for k in ("persistence", "model_b", "model_c"):
    if k in ev:
        print(k, {kk: ev[k][kk] for kk in ("mae", "rmse", "spatial_correlation")})

In [ ]:
from IPython.display import Image as IImage, display
display(IImage(filename=str(BACKEND / cfg.out_dir / "evaluation/forecast_maps.png")))

## 7. Inference: predict the next day's sea-ice concentration

In [ ]:
from ml.sea_ice.predict import run as predict_run
pred = predict_run(Path(cfg.out_dir))
print(json.dumps({k: pred[k] for k in ("target_date", "model_name", "output_nc", "statistics")},
                 indent=2))

In [ ]:
import xarray as xr
fig, ax = plt.subplots(figsize=(10, 4))
dsp = xr.open_dataset(pred["output_nc"])
dsp["sea_ice_concentration"].plot(ax=ax, cmap="viridis")
ax.set_title(f"Predicted sea-ice concentration - {pred['target_date']}")

## 8. Honest caveatsIf `demo_only` is True (synthetic inputs), every number in this notebook is a **code-path check**, not a statement about actual forecasting skill. Production use requires real NetCDF files dropped into `backend/datasets/processed/` and re-running the data processing + training pipelines without `--demo`.

In [ ]:
if summary["demo_only"]:
    print(summary["disclaimer"])